In [ ]:
import os, re, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gaussian_kde, mannwhitneyu, linregress
import matplotlib.ticker as mticker

warnings.filterwarnings('ignore')
BASE_DIR = Path(r'')
RAW_DATA = BASE_DIR / 'Data' / 'dataset_ftp_04032026.json'
FIG_DIR  = BASE_DIR / 'Results' / 'Figures' / 'Structural_Individual'
FIG_DIR.mkdir(parents=True, exist_ok=True)

matplotlib.rcParams.update({
    'figure.dpi': 300, 'savefig.dpi': 300,
    'font.family': 'DejaVu Sans', 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3, 'grid.linestyle': '--'
})

PALETTE      = ['#2C6E49', '#C77DFF']
FIELD_COLORS = ['#1B4F72', '#C0392B', '#117A65', '#784212']
FIELDS       = ['inteiro_teor', 'fato', 'direito', 'pedido']
FIELDS_LABEL = ['Full Text', 'Facts', 'Legal Basis', 'Legal Claim']

print('Loading data...')
with open(RAW_DATA, 'r', encoding='utf-8', errors='replace') as f:
    df = pd.DataFrame(json.load(f))
df['is_recurso'] = df['is_recurso'].astype(bool)
    
for fld in FIELDS:
    df[f'{fld}_words'] = df[fld].astype(str).str.split().str.len()
    df[f'{fld}_chars'] = df[fld].astype(str).str.len()
    
print(f'Loaded {len(df)} records. Setup complete.')

## Section 1: Word Count Histograms


In [ ]:
for fld, lbl, color in zip(FIELDS, FIELDS_LABEL, FIELD_COLORS):
    fig, ax = plt.subplots(figsize=(6, 4.5))
    col = f'{fld}_words'
    data = df[col]
    p99 = data.quantile(0.99)
    data_clp = data.clip(upper=p99)
    
    ax.hist(data_clp, bins=45, color=color, alpha=0.6, density=True, edgecolor='white')
    
    kde = gaussian_kde(data_clp)
    xr = np.linspace(data_clp.min(), data_clp.max(), 300)
    ax.plot(xr, kde(xr), color=color, lw=2)
    
    ax.axvline(data.median(), color='#444444', linestyle='--', lw=1.5, label=f'Median: {data.median():.0f}')
    
    ax.set_title(f'Word Count Distribution: {lbl}', weight='bold')
    ax.set_xlabel('Word Count (clipped at 99th percentile)')
    ax.set_ylabel('Density')
    ax.legend()
    
    p = FIG_DIR / f'1_wordcount_hist_{fld}.png'
    fig.savefig(p, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f'Exported -> {p.name}')

## Section 2: Character Counts (Violin & Box)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
clipped = [df[f'{f}_chars'].clip(upper=df[f'{f}_chars'].quantile(0.99)) for f in FIELDS]
parts = ax.violinplot(clipped, showmedians=True)
for j, pc in enumerate(parts['bodies']):
    pc.set_facecolor(FIELD_COLORS[j]); pc.set_alpha(0.7)
parts['cmedians'].set_color('#111111')
ax.set_xticks(range(1, len(FIELDS)+1)); ax.set_xticklabels(FIELDS_LABEL)
ax.set_ylabel('Character Count (P99 clip)')
ax.set_title('Character Count Distribution (Violin)', weight='bold')
p = FIG_DIR / '2_charcount_violin.png'
fig.savefig(p, bbox_inches='tight')
plt.close()
print(f'Exported -> {p.name}')

fig, ax = plt.subplots(figsize=(8, 5))
bp = ax.boxplot([df[f'{f}_chars'] for f in FIELDS], patch_artist=True, showfliers=False)
for patch, color in zip(bp['boxes'], FIELD_COLORS):
    patch.set_facecolor(color); patch.set_alpha(0.7)
for med in bp['medians']: med.set_color('black')
ax.set_yscale('log')
ax.set_xticks(range(1, len(FIELDS)+1)); ax.set_xticklabels(FIELDS_LABEL)
ax.set_ylabel('Character Count (Log Scale)')
ax.set_title('Character Count Distribution (Box)', weight='bold')
p = FIG_DIR / '3_charcount_box.png'
fig.savefig(p, bbox_inches='tight')
plt.close()
print(f'Exported -> {p.name}')

## Section 3: Stratified by Document Type (Appeal vs Non-Appeal)


In [ ]:
g_false = df[df.is_recurso == False]
g_true  = df[df.is_recurso == True]

for fld, lbl in zip(FIELDS, FIELDS_LABEL):
    fig, ax = plt.subplots(figsize=(5, 5))
    col = f'{fld}_words'
    p99 = df[col].quantile(0.99)
    d0 = g_false[col].clip(upper=p99)
    d1 = g_true[col].clip(upper=p99)
    
    bp = ax.boxplot([d0, d1], patch_artist=True, labels=['Non-Appeal\n(First Instance)', 'Appeal\n(Second Instance)'], showfliers=False)
    for patch, color in zip(bp['boxes'], PALETTE):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    for med in bp['medians']: med.set_color('black')
        
    u_stat, p_val = mannwhitneyu(g_false[col], g_true[col], alternative='two-sided')
    
    ax.set_ylabel('Word Count (P99 clip)')
    ax.set_title(f'{lbl} Length Variation', weight='bold')
    ax.text(0.5, 0.95, f'Mann-Whitney p-val: {p_val:.2e}', transform=ax.transAxes, ha='center', va='top', bbox=dict(fc='white', alpha=0.8))
    
    p = FIG_DIR / f'4_stratified_length_{fld}.png'
    fig.savefig(p, bbox_inches='tight')
    plt.close()
    print(f'Exported -> {p.name}')

## Section 4: Privacy / PII Surface Area


In [ ]:
PII_PATTERNS = {
    'CPF (Individual Tax ID)': r'\d{3}\.\d{3}\.\d{3}-\d{2}',
    'CNPJ (Company Tax ID)': r'\d{2}\.\d{3}\.\d{3}/\d{4}-\d{2}',
    'Process Number': r'\d{7}-\d{2}\.\d{4}\.\d\.\d{2}\.\d{4}'
}

totals = {}
for name, pat in PII_PATTERNS.items():
    rgx = re.compile(pat)
    totals[name] = df['inteiro_teor'].astype(str).str.contains(rgx).sum()

fig, ax = plt.subplots(figsize=(6, 4))
colors = ['#E74C3C', '#3498DB', '#F39C12']
bars = ax.barh(list(totals.keys()), list(totals.values()), color=colors, alpha=0.8)
for bar in bars:
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2, 
            f'{int(bar.get_width())} ({int(bar.get_width())/len(df)*100:.1f}%)', 
            va='center', weight='bold')

ax.set_xlim(0, max(totals.values()) * 1.3)
ax.set_xlabel('Documents Containing PII Pattern (out of 1,000)')
ax.set_title('PII Exposure Rate', weight='bold')
ax.invert_yaxis()
p = FIG_DIR / '5_pii_exposure.png'
fig.savefig(p, bbox_inches='tight')
plt.close()
print(f'Exported -> {p.name}')

## Section 5: Argumentative Ratios


In [ ]:
df['ratio_claim_basis'] = df['pedido_words'] / (df['direito_words'] + 1)
df['ratio_facts_basis'] = df['fato_words'] / (df['direito_words'] + 1)

for col, title, fname in [
    ('ratio_claim_basis', 'Claim / Basis Density Ratio', '6a_ratio_claim_basis.png'),
    ('ratio_facts_basis', 'Facts / Basis Density Ratio', '6b_ratio_facts_basis.png')
]:
    fig, ax = plt.subplots(figsize=(6, 4.5))
    
    p99 = df[col].quantile(0.99)
    d_false = df[df.is_recurso==False][col].clip(upper=p99)
    d_true  = df[df.is_recurso==True][col].clip(upper=p99)
    
    ax.hist(d_false, bins=35, color=PALETTE[0], alpha=0.6, label='Non-Appeal', density=True)
    ax.hist(d_true, bins=35, color=PALETTE[1], alpha=0.6, label='Appeal', density=True)
    
    ax.set_title(title, weight='bold')
    ax.set_xlabel('Ratio value (clipped at P99)')
    ax.axvline(1.0, color='black', ls='--')
    ax.legend()
    
    p = FIG_DIR / fname
    fig.savefig(p, bbox_inches='tight')
    plt.close()
    print(f'Exported -> {p.name}')

## Section 6: Legal Marker Frequency


In [ ]:
LEGAL_MARKERS = {'Art.': r'\bart\.\s*\d+', 'STJ': r'\bstj\b', 'STF': r'\bstf\b', 
                 'CPC': r'\bcpc\b', 'CDC': r'\bcdc\b', 'Moral Damages': r'danos?\s+morais?'}

marker_total = {}
for name, pat in LEGAL_MARKERS.items():
    rgx = re.compile(pat, re.IGNORECASE)
    marker_total[name] = df['direito'].astype(str).str.contains(rgx).sum()

sorted_names = sorted(marker_total, key=lambda k: marker_total[k], reverse=True)
vals = [marker_total[m] for m in sorted_names]

fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.barh(sorted_names, vals, color='#34495E', alpha=0.8)
for bar in bars:
    w = bar.get_width()
    ax.text(w + 5, bar.get_y() + bar.get_height()/2, f'{int(w)}', va='center')
ax.set_xlabel('Documents Containing Marker (Legal Basis Field)')
ax.set_title('Most Frequent Legal Markers', weight='bold')
ax.invert_yaxis()
p = FIG_DIR / '7_legal_markers.png'
fig.savefig(p, bbox_inches='tight')
plt.close()
print(f'Exported -> {p.name}')

## Section 7: Deep Scatter (Basis vs Claim)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5.5))

for val, lbl, color, m in [(False,'Non-Appeal',PALETTE[0],'o'), (True,'Appeal',PALETTE[1],'^')]:
    sub = df[df.is_recurso == val]
    ax.scatter(sub['direito_words']+1, sub['pedido_words']+1, 
               c=color, marker=m, alpha=0.35, label=lbl, s=25, edgecolor='none')

ax.set_xscale('log'); ax.set_yscale('log')
ax.plot([1, 10000], [1, 10000], 'k--', lw=1.5, alpha=0.5, label='Equal Length (y=x)')
ax.set_xlabel('Legal Basis Word Count (Log Scale)')
ax.set_ylabel('Legal Claim Word Count (Log Scale)')
ax.set_title('Cross-Field Length Asymmetry', weight='bold')
ax.legend()

p = FIG_DIR / '8_cross_scatter_log.png'
fig.savefig(p, bbox_inches='tight')
plt.close()
print(f'Exported -> {p.name}')

## Section 8: Correlation Heatmap


In [ ]:
cols = [f'{f}_words' for f in FIELDS]
corr = df[cols].rename(columns={f'{f}_words': l for f, l in zip(FIELDS, FIELDS_LABEL)}).corr(method='spearman')
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, mask=mask, ax=ax, annot=True, fmt='.2f', cmap='RdBu_r', vmin=-1, vmax=1, 
            square=True, cbar_kws={'label': 'Spearman Correlation'})
ax.set_title('Field Length Correlation Matrix', weight='bold', pad=15)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
p = FIG_DIR / '9_correlation_heatmap.png'
fig.savefig(p, bbox_inches='tight')
plt.close()
print(f'Exported -> {p.name}')
print('\nALL INDIVIDUAL FIGURE EXPORTS COMPLETE.')

In [ ]:
STRUCT_RESULTS_DIR = BASE_DIR / 'Results'
STRUCT_TABLE_DIR   = STRUCT_RESULTS_DIR / 'Tables'
STRUCT_TABLE_DIR.mkdir(parents=True, exist_ok=True)

summary = {
    'n_total': int(len(df)),
    'n_non_appeal': int((df['is_recurso'] == False).sum()),
    'n_appeal': int((df['is_recurso'] == True).sum()),
    'fields': {}
}

for fld, lbl in zip(FIELDS, FIELDS_LABEL):
    words = df[f'{fld}_words']
    chars = df[f'{fld}_chars']
    summary['fields'][fld] = {
        'label': lbl,
        'words_mean': float(words.mean()),
        'words_median': float(words.median()),
        'words_std': float(words.std()),
        'words_p95': float(words.quantile(0.95)),
        'chars_mean': float(chars.mean()),
        'chars_median': float(chars.median()),
        'chars_std': float(chars.std()),
        'chars_p95': float(chars.quantile(0.95)),
    }

rows = []
for fld, meta in summary['fields'].items():
    rows.append({
        'field': fld,
        'label': meta['label'],
        'words_mean': round(meta['words_mean'], 2),
        'words_median': round(meta['words_median'], 2),
        'words_std': round(meta['words_std'], 2),
        'words_p95': round(meta['words_p95'], 2),
        'chars_mean': round(meta['chars_mean'], 2),
        'chars_median': round(meta['chars_median'], 2),
        'chars_std': round(meta['chars_std'], 2),
        'chars_p95': round(meta['chars_p95'], 2),
    })

df_struct = pd.DataFrame(rows)
csv_path  = STRUCT_TABLE_DIR / 'structural_dataset_card.csv'
json_path = STRUCT_RESULTS_DIR / 'structural_dataset_card.json'

df_struct.to_csv(csv_path, index=False, encoding='utf-8-sig')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print('\n=== STRUCTURAL DATASET CARD ===')
print(df_struct.to_string(index=False))
print(f'\nSaved CSV  -> {csv_path}')
print(f'Saved JSON -> {json_path}')